# DS extractor smoke test — `esn_identifier`

Self-contained Databricks notebook to run the DS team's `esn_identifier` (LLM-count extractor) against a small dev cohort and compare with the currently stored `esn` value in dev metadata.

**Read-only.** No Delta writes, no VS index changes. Calls the LiteLLM dev gateway and reads PDFs from dev volumes.


## 0. Install dependencies

Run once. `dbutils.library.restartPython()` restarts the Python kernel so the new packages are picked up — **all later cells must be re-run after this**.


In [ ]:
%pip install pymupdf httpx --quiet
dbutils.library.restartPython()


## 1. Config — edit these before running

- `LITELLM_BASE_URL` / `LITELLM_API_KEY` — LiteLLM gateway endpoint and key. If a Databricks secret scope is set up, prefer `dbutils.secrets.get(...)`.
- `METADATA_TABLE` — dev metadata table to resolve `volume_path` for each cohort doc.
- `DOC_IDS` — small cohort of `document_id` values to test against.
- `OUTPUT_CSV_PATH` — writable Volume path. Set to `None` to skip writing (display only).


In [ ]:
# === EDIT ME ===
LITELLM_BASE_URL = "https://dev-gateway.apps.gevernova.net"
LITELLM_API_KEY  = ""  # paste dev key, or: dbutils.secrets.get("<scope>", "<key>")

METADATA_TABLE = "vaid.ai_sot_field_service_report.biz_metadata_field_service_report"

OUTPUT_CSV_PATH = None  # e.g. "/Volumes/vaid/.../esn_test/comparison.csv" or None to skip

DOC_IDS = [
    # Cohort under test — mix of fieldvision UUIDs and manual ecrt report ids
    "d3b1da8a-03b4-4ea6-aa7b-0482edb532ce",
    "3ef8d150-1225-4c62-ac5e-ae9166c73954",
    "913433ad-6d19-472f-917c-5048f689a5ed",
    "c020379c-93a4-49f6-b6a9-1388edd59d3a",
    "03d3bef2-db2b-4311-b147-641025e3636b",
    "36de1c13-acee-4633-8b76-8a262f738a20",
    "34bc62b0-573c-4f43-bc62-b0573ccf43e9",
    "348ddaa2-8c32-4646-9f17-c57f33256d82",
    "4b70aab1-e210-4d5b-9cbf-bfe690436861",
    "d5334dce-9d7f-4fa1-b34d-ce9d7f7fa130",
    "090dbba1800f4f5b",
    "090dbba18003a23c",
    "090dbba180100488",
    "090dbba1800b5e28",
]

# Extractor thresholds
# NOTE: DS defaults (5, 0.10) qualify only ~3/14 docs in our cohort because the
# prepared text window (~3010 chars) limits how many times an ESN can appear.
# For a more representative comparison try MIN_ESN_COUNT=1, MIN_ESN_FRACTION=0.10
# — top-1 by frequency then matches stored ESN on every successful UUID doc.
MIN_ESN_COUNT    = 5     # try 1 to compare top-1-by-frequency against stored
MIN_ESN_FRACTION = 0.10  # keep at 0.10
ESN_LLM_MODEL    = "azure-gpt-5-2"   # options: "azure-gpt-5-2" or "gemini-3-flash" (reasoning model — needs ~4000 tokens)

print(f"Cohort size: {len(DOC_IDS)}")
print(f"Metadata table: {METADATA_TABLE}")
print(f"LiteLLM URL: {LITELLM_BASE_URL}")
print(f"API key set: {bool(LITELLM_API_KEY)}")


## 2. Inlined DS code

Below cells inline the minimum DS code needed to call `esn_identifier`:
- `pymupdf_guard` — thread lock around fitz calls
- `utils.setup_logger` — logger helper
- A stripped `config` shim — exposes `LITELLM_BASE_URL`, `LITELLM_API_KEY`, `SECRET_SCOPE`, `CERT_PATH`, `CORP_PROXY` from the values you set in the config cell above
- `esn_identifier` — full module, imports rewritten to use the locals above

No file uploads needed. Just run all.


### 2a. `pymupdf_guard` — thread-lock for PyMuPDF


In [ ]:
# Inlined from DS-ref-code/.../src/pymupdf_guard.py
from contextlib import contextmanager
import os, threading

_PYMUPDF_LOCK = threading.Lock()

@contextmanager
def hold_pymupdf_lock():
    use_lock = os.getenv("FSR_USE_PYMUPDF_LOCK", "true").strip().lower() not in {"0","false","f","no","n","off"}
    if not use_lock:
        yield
        return
    with _PYMUPDF_LOCK:
        yield


### 2b. `setup_logger` (from utils)


In [ ]:
# Inlined from DS-ref-code/.../src/utils.py
import logging

def setup_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    logger.addHandler(ch)
    return logger


### 2c. `config` shim

Strips DS `config.py` down to the 5 names `esn_identifier` actually imports. Pulls values from the config cell at top of notebook. Cert / proxy disabled — your dev cluster already trusts the gateway.


In [ ]:
# Stripped config — exposes only what esn_identifier imports
SECRET_SCOPE = ""           # not used in test path
CERT_PATH    = None         # cluster already trusts dev gateway
CORP_PROXY   = ""           # no proxy on dbr cluster
# LITELLM_BASE_URL and LITELLM_API_KEY come from the top config cell


### 2d. `esn_identifier` — full module, inlined

Same code as the DS team's `esn_identifier.py`, with the 3 cross-module imports (`config`, `pymupdf_guard`, `utils`) removed — those names already exist in this notebook from cells above.

Public functions exposed after this cell:
- `load_document_text_for_esn(pdf_path)` — extracts full PDF text via PyMuPDF
- `prepare_document_text_for_esn(text)` — start/middle/end windowing if too long
- `analyze_prepared_document_text_for_esn_counts(text)` — one LLM call → `{esn: count}`
- `analyze_pdf_for_esn_counts(pdf_path)` — convenience wrapper of all 3
- `_qualify_esn_counts(counts, MIN_ESN_COUNT, MIN_ESN_FRACTION)` — apply thresholds → ranked list of qualified ESNs


In [ ]:
"""
ESN Identification – one document-level LLM count call with uniform chunk tags.

The document is analyzed once, qualified ESNs are selected using the global
count and fraction thresholds, and every chunk in the document receives the
same ESN label set.
"""
import json
import logging
import os
import re
import time
from pathlib import Path
from typing import Any, Dict, List, Optional
from urllib.parse import urlparse

import fitz
import httpx


logger = setup_logger("esn_identifier")
logger.setLevel(logging.WARNING)

ESN_LLM_MODEL = os.getenv("ESN_LLM_MODEL", "gemini-3-flash")
ESN_LLM_MAX_RETRIES = int(os.getenv("ESN_LLM_MAX_RETRIES", "5"))
ESN_LLM_RETRY_BASE_DELAY_SEC = float(os.getenv("ESN_LLM_RETRY_BASE_DELAY_SEC", "2.0"))
MIN_ESN_COUNT = 5
MIN_ESN_FRACTION = 0.10
DOC_TRUNCATE_WINDOW_CHARS = 1000
DOC_MAX_LLM_CHARS = DOC_TRUNCATE_WINDOW_CHARS * 3

_VALID_ESN_RE = re.compile(r"^[A-Z0-9]{4,12}$", re.IGNORECASE)
_TRANSIENT_STATUS_CODES = {429, 500, 502, 503, 504}

_DOC_SYSTEM_PROMPT = """\
You identify equipment serial numbers (ESNs) in field service report text.

Rules:
- An ESN is a 4-12 character alphanumeric code identifying a gas turbine or generator.
- Count only explicit ESN mentions across the full document.
- Keys in the JSON response must be the ESN values themselves.
- Values must be integer mention counts.
- Do not infer missing ESNs.
- Do not return part numbers, ER case numbers, dates, TIL numbers, page numbers, or other non-ESN identifiers.
- If no ESNs are explicitly mentioned, return {}.

Respond with ONLY a JSON object, for example {"297837": 7, "290T658": 2}.
No markdown. No prose. No explanation.
"""

_DOC_USER_TEMPLATE = """\
Count explicit ESN mentions in this field service report.

Return ONLY a JSON object whose keys are ESNs and whose values are integer counts.
Return all explicit ESN counts you can find across the full document, including counts below 5.
The downstream pipeline applies the minimum-count and percentage thresholds.

Document text:
{document_text}
"""


def _build_http_client() -> httpx.Client:
    """Build an httpx client respecting corporate proxy/cert settings."""
    verify = str(CERT_PATH) if CERT_PATH and Path(CERT_PATH).exists() else True
    base_host = (urlparse(LITELLM_BASE_URL).hostname or "").lower()
    bypass_proxy = base_host.endswith("research.gevernova.net")
    client_kwargs = {"verify": verify, "timeout": 120.0, "trust_env": False}
    if CORP_PROXY and not bypass_proxy:
        try:
            return httpx.Client(proxy=CORP_PROXY, **client_kwargs)
        except TypeError:
            return httpx.Client(proxies=CORP_PROXY, **client_kwargs)
    return httpx.Client(**client_kwargs)


def _strip_markdown_fences(text: str) -> str:
    t = (text or "").strip()
    if not t.startswith("```"):
        return t

    parts = t.split("```")
    if len(parts) >= 3:
        inner = parts[1].strip()
        if inner.lower().startswith("json"):
            inner = inner[4:].strip()
        return inner
    return t


def _extract_first_json_container(text: str) -> str:
    start = None
    stack: List[str] = []
    in_string = False
    escaped = False
    pairs = {"{": "}", "[": "]"}
    closers = set(pairs.values())

    for i, ch in enumerate(text):
        if start is None:
            if ch in pairs:
                start = i
                stack = [ch]
                in_string = False
                escaped = False
            continue

        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch in pairs:
            stack.append(ch)
        elif ch in closers and stack:
            opener = stack[-1]
            if pairs.get(opener) == ch:
                stack.pop()
            if not stack:
                return text[start:i + 1]

    raise ValueError("No balanced JSON object/array found in LLM response")


def _repair_truncated_json_object(text: str) -> str:
    start = text.find("{")
    if start < 0:
        raise ValueError("No opening '{' found for truncated JSON repair")

    depth = 0
    in_string = False
    escaped = False
    last_top_level_comma = -1

    for i, ch in enumerate(text[start:], start=start):
        if in_string:
            if escaped:
                escaped = False
            elif ch == "\\":
                escaped = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch in "{[":
            depth += 1
        elif ch in "}]":
            depth = max(0, depth - 1)
        elif ch == "," and depth == 1:
            last_top_level_comma = i

    if last_top_level_comma < 0:
        raise ValueError("No top-level comma found for truncated JSON repair")

    repaired = text[start:last_top_level_comma].rstrip()
    if repaired.endswith(","):
        repaired = repaired[:-1].rstrip()
    repaired += "}"
    return repaired


def _normalize_esn_token(value: Any) -> Optional[str]:
    if not isinstance(value, str):
        return None

    token = value.strip().strip('"\'[]()')
    if not token:
        return None

    token = token.upper()
    if not _VALID_ESN_RE.match(token):
        return None
    return token


def _coerce_positive_int(value: Any) -> Optional[int]:
    if isinstance(value, bool):
        return None
    if isinstance(value, int):
        return value if value > 0 else None
    if isinstance(value, float):
        return int(round(value)) if value > 0 else None
    if isinstance(value, str):
        try:
            parsed = float(value.strip())
        except (TypeError, ValueError):
            return None
        return int(round(parsed)) if parsed > 0 else None
    return None


def _normalize_esn_count_payload(parsed: Any) -> Dict[str, int]:
    if not isinstance(parsed, dict):
        raise ValueError("Expected JSON object mapping ESN -> count")

    counts: Dict[str, int] = {}
    for raw_esn, raw_count in parsed.items():
        esn = _normalize_esn_token(raw_esn)
        count = _coerce_positive_int(raw_count)
        if not esn or count is None:
            continue
        counts[esn] = counts.get(esn, 0) + count
    return counts


def _parse_esn_count_response(raw: str) -> Dict[str, int]:
    cleaned = _strip_markdown_fences(raw)

    try:
        json_text = _extract_first_json_container(cleaned)
        parsed = json.loads(json_text)
        return _normalize_esn_count_payload(parsed)
    except (json.JSONDecodeError, TypeError, ValueError):
        pass

    try:
        repaired = _repair_truncated_json_object(cleaned)
        parsed = json.loads(repaired)
        return _normalize_esn_count_payload(parsed)
    except (json.JSONDecodeError, TypeError, ValueError):
        pass

    normalized = cleaned.strip().lower()
    if normalized in {"", "{}", "[]", "none", "null", "no esn", "no esns"}:
        return {}

    raise ValueError("Unable to parse ESN count response as JSON object")


def _normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def _prepare_document_text_for_llm(text: str) -> str:
    """Keep the full document when possible, otherwise use start/middle/end windows."""
    normalized = _normalize_text(text)
    if len(normalized) <= DOC_MAX_LLM_CHARS:
        return normalized

    window = DOC_TRUNCATE_WINDOW_CHARS
    middle_start = max(
        window,
        min((len(normalized) // 2) - (window // 2), len(normalized) - (2 * window)),
    )
    middle_end = middle_start + window

    return "\n...\n".join([
        normalized[:window],
        normalized[middle_start:middle_end],
        normalized[-window:],
    ])


def prepare_document_text_for_esn(document_text: str) -> str:
    """Prepare the preloaded document text once before dispatching LLM work."""
    return _prepare_document_text_for_llm(document_text)


def _extract_document_text(pdf_path: str) -> str:
    with hold_pymupdf_lock():
        try:
            doc = fitz.open(pdf_path)
        except Exception as exc:
            raise RuntimeError(f"Failed to open PDF for ESN text preload: {exc}") from exc

        try:
            pages = []
            for page in doc:
                page_text = _normalize_text(page.get_text("text"))
                if page_text:
                    pages.append(page_text)
            return "\n\n".join(pages)
        finally:
            doc.close()


def load_document_text_for_esn(pdf_path: str) -> str:
    """Load the full document text once on the caller thread."""
    return _extract_document_text(pdf_path)


def analyze_prepared_document_text_for_esn_counts(prepared_document_text: str) -> Dict[str, int]:
    """Run one LLM call over already-prepared document text."""
    if not LITELLM_API_KEY or not LITELLM_API_KEY.strip():
        raise RuntimeError(
            f"[ESN-LLM] Missing LITELLM_API_KEY in secret scope '{SECRET_SCOPE}'."
        )

    if not prepared_document_text:
        return {}

    url = LITELLM_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {LITELLM_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": ESN_LLM_MODEL,
        "messages": [
            {"role": "system", "content": _DOC_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": _DOC_USER_TEMPLATE.format(document_text=prepared_document_text),
            },
        ],
        "max_completion_tokens": 4000,
        "temperature": 0,
    }
    raw = ""
    last_error: BaseException | None = None

    with _build_http_client() as client:
        for attempt in range(1, ESN_LLM_MAX_RETRIES + 1):
            raw = ""
            try:
                resp = client.post(url, headers=headers, json=payload)
                if resp.status_code != 200:
                    error = RuntimeError(f"HTTP {resp.status_code}: {resp.text[:200]}")
                    if resp.status_code in _TRANSIENT_STATUS_CODES and attempt < ESN_LLM_MAX_RETRIES:
                        time.sleep(ESN_LLM_RETRY_BASE_DELAY_SEC * attempt)
                        last_error = error
                        continue
                    raise error

                raw = resp.json()["choices"][0]["message"]["content"].strip()
                return _parse_esn_count_response(raw)
            except (json.JSONDecodeError, KeyError, TypeError, ValueError) as exc:
                preview = (raw or "")[:5000].replace("\n", "\\n")
                raise RuntimeError(
                    f"[ESN-LLM] Failed to parse doc-level response: {exc}. Raw preview: {preview}"
                ) from exc
            except (httpx.TimeoutException, httpx.NetworkError, httpx.RemoteProtocolError) as exc:
                if attempt < ESN_LLM_MAX_RETRIES:
                    time.sleep(ESN_LLM_RETRY_BASE_DELAY_SEC * attempt)
                    last_error = exc
                    continue
                last_error = exc
                break
            except BaseException as exc:
                last_error = exc
                break

    if last_error is None:
        last_error = RuntimeError("Unknown doc-level LLM failure")
    raise RuntimeError(
        f"[ESN-LLM] Doc-level call failed ({type(last_error).__name__}): {last_error}"
    ) from last_error


def analyze_document_text_for_esn_counts(document_text: str) -> Dict[str, int]:
    """Run one LLM call over already-loaded document text."""
    return analyze_prepared_document_text_for_esn_counts(
        prepare_document_text_for_esn(document_text)
    )


def analyze_pdf_for_esn_counts(pdf_path: str) -> Dict[str, int]:
    """Run one LLM call over the full document and return raw ESN counts."""
    return analyze_document_text_for_esn_counts(load_document_text_for_esn(pdf_path))


def _qualify_esn_counts(
    counts: Dict[str, int],
    min_count: int,
    min_fraction: float,
) -> List[str]:
    total_mentions = sum(counts.values())
    if total_mentions <= 0:
        return []

    ranked = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [
        esn
        for esn, count in ranked
        if count >= min_count and (count / total_mentions) >= min_fraction
    ]


def _fallback_generator_serial(meta: Dict[str, Any]) -> List[str]:
    fallback = _normalize_esn_token(meta.get("generator_serial"))
    return [fallback] if fallback else []


def identify_esns(
    chunks: List[Dict],
    doc_counts: Optional[Dict[str, int]] = None,
    min_count: int = MIN_ESN_COUNT,
    min_fraction: float = MIN_ESN_FRACTION,
) -> List[Dict]:
    """Apply one document-level ESN label set to every chunk."""
    if not chunks:
        return chunks

    qualified_doc_esns = _qualify_esn_counts(doc_counts or {}, min_count, min_fraction)

    for chunk in chunks:
        meta = chunk.setdefault("metadata", {})

        if qualified_doc_esns:
            final_labels = list(qualified_doc_esns)
            meta["esn_assignment_scope"] = "document"
        else:
            final_labels = _fallback_generator_serial(meta)
            meta["esn_assignment_scope"] = "fallback" if final_labels else "none"

        meta["chunk_esns"] = list(final_labels)
        meta["esn_labels"] = list(final_labels)
        if final_labels:
            meta["generator_serial"] = final_labels[0]

    return chunks


## 3. Resolve cohort to volume paths

Look up each `document_id` in dev metadata and capture the current `esn`, `esn_source`, and `volume_path`. Any UUID missing here = signal (not present in dev catalog).


In [ ]:
from pyspark.sql import functions as F

meta_df = (
    spark.table(METADATA_TABLE)
         .filter(F.col("document_id").isin(DOC_IDS))
         .select("document_id", "volume_path", "esn", "esn_source", "metadata_status")
         .toPandas()
         .set_index("document_id")
)

resolved   = list(meta_df.index)
unresolved = [d for d in DOC_IDS if d not in resolved]
print(f"Resolved {len(resolved)}/{len(DOC_IDS)} docs in {METADATA_TABLE}")
if unresolved:
    print(f"Missing in dev metadata ({len(unresolved)}): {unresolved}")
display(meta_df)


## 4. Run DS `esn_identifier` over the cohort

Per doc:
1. Load full PDF text via PyMuPDF (`load_document_text_for_esn`)
2. Apply start/middle/end windowing if too long (`prepare_document_text_for_esn`)
3. One LLM call → `{esn: count}` (`analyze_prepared_document_text_for_esn_counts`)
4. Apply `MIN_ESN_COUNT` + `MIN_ESN_FRACTION` thresholds (`_qualify_esn_counts`) — see config cell; **DS defaults (5, 0.10) are strict; lower `MIN_ESN_COUNT` to 1 to surface top-1 candidates**

Logs one line per doc. Sleeps 0.5s between calls to be nice to the gateway.


In [ ]:
# === LLM response shape probe ===
# Send one tiny request to see the raw JSON shape returned by the chosen model.
import httpx, json as _json

_url = LITELLM_BASE_URL.rstrip("/") + "/chat/completions"
_headers = {"Authorization": f"Bearer {LITELLM_API_KEY}", "Content-Type": "application/json"}
_payload = {
    "model": ESN_LLM_MODEL,
    "messages": [
        {"role": "system", "content": "Return ONLY the JSON object: {\"a\": 1}"},
        {"role": "user", "content": "ping"},
    ],
    "max_completion_tokens": 50,
    "temperature": 0,
}
with httpx.Client(verify=False, timeout=30.0, trust_env=False) as _c:
    _r = _c.post(_url, headers=_headers, json=_payload)
print("status:", _r.status_code)
print("body:")
print(_json.dumps(_r.json(), indent=2)[:2000])


In [ ]:
import json as _json, traceback, time

# Push the LITELLM credentials from the top config cell into the inlined module's
# global namespace so analyze_prepared_document_text_for_esn_counts() picks them up.
import sys as _sys
_mod = _sys.modules[__name__]
_mod.LITELLM_BASE_URL = LITELLM_BASE_URL
_mod.LITELLM_API_KEY  = LITELLM_API_KEY

# Tighten retries + timeout
_mod.ESN_LLM_MODEL = ESN_LLM_MODEL
_mod.ESN_LLM_MAX_RETRIES = 2
_mod.ESN_LLM_RETRY_BASE_DELAY_SEC = 1.0

import httpx as _httpx
import warnings as _warnings
try:
    import urllib3 as _urllib3
    _urllib3.disable_warnings(_urllib3.exceptions.InsecureRequestWarning)
except Exception:
    pass
_warnings.filterwarnings("ignore", message="Unverified HTTPS request")
def _build_http_client_short():
    # Match prod behavior on dev gateway: LLM_VERIFY_SSL=False
    return _httpx.Client(verify=False, timeout=30.0, trust_env=False)
_mod._build_http_client = _build_http_client_short
print("[setup] LLM timeout=30s, max_retries=2, verify_ssl=False")

results = []
for i, doc_id in enumerate(DOC_IDS):
    t_start = time.time()
    print(f"[{i+1}/{len(DOC_IDS)}] {doc_id}  starting...", flush=True)
    if doc_id not in meta_df.index:
        print("  -> not in dev metadata, skipping", flush=True)
        results.append({
            "document_id": doc_id, "volume_path": None,
            "current_stored_esn": None, "current_stored_esn_source": None,
            "metadata_status": None,
            "doc_text_chars": None, "llm_raw_counts": None,
            "llm_qualified_esns": None, "llm_primary_esn": None,
            "llm_n_qualified": None,
            "error": "not in dev metadata",
        })
        continue

    row = meta_df.loc[doc_id]
    vp  = row["volume_path"]
    rec = {
        "document_id": doc_id, "volume_path": vp,
        "current_stored_esn": row["esn"],
        "current_stored_esn_source": row["esn_source"],
        "metadata_status": row["metadata_status"],
        "doc_text_chars": None, "llm_raw_counts": None,
        "llm_qualified_esns": None, "llm_primary_esn": None,
        "llm_n_qualified": None, "error": None,
    }
    try:
        print(f"  -> reading PDF: {vp}", flush=True)
        t0 = time.time()
        full_text = load_document_text_for_esn(vp)
        rec["doc_text_chars"] = len(full_text)
        dt = time.time() - t0
        print(f"  -> read {len(full_text)} chars in {dt:.1f}s", flush=True)

        prepared = prepare_document_text_for_esn(full_text)
        truncated = len(prepared) < len(full_text)
        print(f"  -> prepared {len(prepared)} chars (truncated={truncated}); calling LLM...", flush=True)
        t0 = time.time()
        counts = analyze_prepared_document_text_for_esn_counts(prepared)
        dt = time.time() - t0
        print(f"  -> LLM returned {len(counts)} candidates in {dt:.1f}s: {counts}", flush=True)

        qualified = _qualify_esn_counts(counts, MIN_ESN_COUNT, MIN_ESN_FRACTION)
        rec["llm_raw_counts"]     = _json.dumps(counts)
        rec["llm_qualified_esns"] = _json.dumps(qualified)
        rec["llm_primary_esn"]    = qualified[0] if qualified else None
        rec["llm_n_qualified"]    = len(qualified)
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
        rec["error"] = err
        print(f"  -> ERROR: {err}", flush=True)

    elapsed = time.time() - t_start
    stored = rec["current_stored_esn"]
    primary = rec["llm_primary_esn"]
    qual = rec["llm_qualified_esns"]
    print(f"  -> done in {elapsed:.1f}s  stored={stored!r}  primary={primary!r}  qualified={qual}", flush=True)
    results.append(rec)
    time.sleep(0.3)

n_err = sum(1 for r in results if r["error"])
print(f"\nProcessed {len(results)} docs ({n_err} errors)")



## 5. Comparison + CSV

One row per doc. Each row captures: what our pipeline currently stores, what the DS extractor returned (raw and after thresholds), and a quick equality check.

### Column reference

| column | meaning |
|---|---|
| `document_id` | doc under test |
| `volume_path` | resolved PDF path on dev volume (from metadata table) |
| `current_stored_esn` | `esn` value our pipeline currently has in dev metadata — i.e. what our existing extractor wrote. Used as the comparison baseline (not ground truth — on redacted manual ecrt docs this is `XXXXXX`) |
| `current_stored_esn_source` | how the stored value got there: `llm` (our pipeline's LLM) or `fsr_pdf_ref` (UI-14 row-duplication path) |
| `metadata_status` | metadata pipeline status for the doc (`completed` etc.) |
| `doc_text_chars` | size of the full PDF text loaded by PyMuPDF (pre-windowing) |
| `llm_raw_counts` | **DS LLM output, before any thresholds.** JSON object `{esn: count}` — every ESN candidate the LLM saw in the prepared text and how many times. Top-1 by count is a reasonable signal even when thresholds reject everything |
| `llm_qualified_esns` | ESNs that survive `_qualify_esn_counts` filter: `count >= MIN_ESN_COUNT` AND `count/total >= MIN_ESN_FRACTION`, ranked by count desc, then alphabetical. With DS defaults (5, 0.10) this is empty for most docs in our cohort — the 3000-char window rarely shows an ESN ≥5 times |
| `llm_primary_esn` | first element of `llm_qualified_esns`, or null. The "ESN we'd actually use" if we adopted DS as-is |
| `llm_n_qualified` | count of qualified ESNs. `0` = null/redacted signal; `>1` = multi-ESN signal |
| `error` | exception message if the LLM call failed for this doc; null on success |
| `llm_eq_stored` | boolean: does `llm_primary_esn` equal `current_stored_esn`? Quick scan column. Note: `False` on redacted manual ecrt docs is **expected and good** (DS recovered a real ESN; stored is the `XXXXXX` placeholder) |



In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df["llm_eq_stored"] = (df["llm_primary_esn"] == df["current_stored_esn"])

display(df)

if OUTPUT_CSV_PATH:
    df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"Wrote: {OUTPUT_CSV_PATH}")
else:
    print("OUTPUT_CSV_PATH is None — display only, no file written.")


## 6. Quick summary

- How many docs did DS LLM agree with the currently stored `esn`?
- How many docs returned 0 qualified ESNs (null/redacted signal)?
- How many docs returned >1 qualified ESN (multi-ESN signal)?


In [ ]:
n = len(df)
n_err   = df["error"].notna().sum()
n_match = df["llm_eq_stored"].sum()
n_zero  = (df["llm_n_qualified"] == 0).sum()
n_multi = (df["llm_n_qualified"] > 1).sum()

print(f"Total docs:              {n}")
print(f"  errors:                {n_err}")
print(f"  LLM primary == stored: {n_match}")
print(f"  LLM 0 qualified:       {n_zero}   (null/redacted signal)")
print(f"  LLM >1 qualified:      {n_multi}   (multi-ESN signal)")

# --- Copy-friendly CSV dump of the comparison dataframe ---
# Prints the full df as CSV text. Select-all + copy out of the cell output to paste
# into a spreadsheet or share. Skips writing to disk (use OUTPUT_CSV_PATH for that).
print("\n" + "=" * 80)
print("CSV (copy from below):")
print("=" * 80)
print(df.to_csv(index=False))
